# ECNet — Dataset Provenance Audit

Recovers **where the training data came from**, for the paper's Data section.

**How to use**
1. New Kaggle Notebook → **Add Input** → attach every dataset you used
   (`hybridframeextraction`, `frames-joman-10000`, `extracted-frames-10000`,
   `extract-frames-gdrive`, plus the original video datasets if you still have them).
2. Run all cells top to bottom.
3. Copy the final table into your paper.

Read-only — nothing is modified.

In [ ]:
# ============================== 1. WHAT IS ATTACHED ==========================
import os, json, collections
from pathlib import Path

ROOT = Path("/kaggle/input")
FILE_CAP = 400_000          # stop counting past this (frame sets are huge)

def walk_stats(root, cap=FILE_CAP):
    """(n_files, total_bytes, ext_counter, sample_names) without exhausting RAM."""
    n, size, exts, sample = 0, 0, collections.Counter(), []
    for dirpath, _dirnames, filenames in os.walk(root):
        for fn in filenames:
            n += 1
            exts[Path(fn).suffix.lower()] += 1
            if len(sample) < 40:
                sample.append(str(Path(dirpath).relative_to(root) / fn))
            if n <= 5000:                      # only stat a prefix, for speed
                try: size += os.path.getsize(os.path.join(dirpath, fn))
                except OSError: pass
            if n >= cap:
                return n, size, exts, sample, True
    return n, size, exts, sample, False

if not ROOT.exists():
    raise SystemExit("No /kaggle/input — attach your datasets via 'Add Input'.")

inputs = sorted(p for p in ROOT.iterdir() if p.is_dir())
print(f"Attached inputs: {len(inputs)}\n")

STATS = {}
for d in inputs:
    n, size, exts, sample, capped = walk_stats(d)
    STATS[d.name] = dict(path=d, n=n, exts=exts, sample=sample, capped=capped)
    top = ", ".join(f"{e or '(none)'}:{c:,}" for e, c in exts.most_common(4))
    print(f"  {d.name:<38} {n:>9,} files{' (capped)' if capped else ''}   {top}")

## 2. Directory structure
Folder names usually encode the class split (`real/` vs `ai_generated/`) and sometimes the source.

In [ ]:
# ============================== 2. STRUCTURE =================================
def tree(root, max_depth=3, max_children=12):
    root = Path(root)
    base = len(root.parts)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = len(Path(dirpath).parts) - base
        if depth > max_depth:
            dirnames[:] = []
            continue
        dirnames.sort()
        indent = "   " * depth
        name = Path(dirpath).name if depth else root.name
        print(f"{indent}{name}/   [{len(filenames):,} files, {len(dirnames)} subdirs]")
        if depth == max_depth:
            dirnames[:] = []
        elif len(dirnames) > max_children:
            print(f"{indent}   ... {len(dirnames)} subdirs, showing {max_children}")
            dirnames[:] = dirnames[:max_children]

for name, s in STATS.items():
    print("=" * 78)
    tree(s["path"])
    print()

## 3. Manifests & READMEs
The most direct evidence: any CSV / JSON / TXT / MD shipped inside a dataset. Frame datasets normally carry an **index CSV** whose columns record the label and often the source.

In [ ]:
# ============================== 3. MANIFESTS =================================
import pandas as pd

DOC_EXT = {".csv", ".json", ".txt", ".md", ".yaml", ".yml"}
docs = []
for name, s in STATS.items():
    for dirpath, _d, filenames in os.walk(s["path"]):
        for fn in filenames:
            if Path(fn).suffix.lower() in DOC_EXT:
                docs.append((name, Path(dirpath) / fn))
        if len(docs) > 300:
            break

print(f"Found {len(docs)} doc/manifest files\n")
for ds, p in docs[:25]:
    print("=" * 78)
    print(f"[{ds}]  {p.name}   ({p.stat().st_size:,} bytes)")
    try:
        if p.suffix.lower() == ".csv":
            df = pd.read_csv(p, nrows=5000)
            print(f"  rows(sampled)={len(df):,}  columns={list(df.columns)}")
            print(df.head(3).to_string(max_colwidth=44))
            # value counts for columns that plausibly encode provenance
            for col in df.columns:
                lc = col.lower()
                if any(k in lc for k in ("label", "class", "source", "split",
                                         "platform", "gen", "orig", "dataset")):
                    vc = df[col].astype(str).value_counts().head(12)
                    print(f"  -- {col} --")
                    for k, v in vc.items():
                        print(f"       {str(k)[:52]:<54}{v:>8,}")
        else:
            print(p.read_text(errors="ignore")[:1200])
    except Exception as e:
        print(f"  (unreadable: {type(e).__name__}: {e})")
    print()

## 4. Filename provenance

`evaluate.py` derives the source from filename prefixes. This counts them for real,
so you learn what each dataset actually contains.

| prefix | source |
|---|---|
| `ugc_` | YouTube UGC |
| `vis_` | vision_devices (phone camera) |
| `pexl_` / `pex_` | Pexels |
| `dact_` | DeepAction |
| `avg_` | AVGen-Bench (generator-tagged) |
| `gvb_` | GenVidBench |
| *(none)* | `stock_pool` / `legacy_pool` — **unrecorded** |

In [ ]:
# ============================== 4. PREFIX TAXONOMY ===========================
PREFIX = [("ugc_", "youtube_ugc"), ("vis_", "vision_devices"),
          ("pexl_", "pexels"), ("pex_", "pexels"), ("dact_", "deepaction"),
          ("avg_", "avgen_bench"), ("gvb_", "genvidbench")]

def source_of(stem):
    for pre, tag in PREFIX:
        if stem.startswith(pre):
            return tag
    return "UNTAGGED (stock_pool / legacy_pool)"

MEDIA_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp",
             ".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v"}

def stems_of(root, cap=FILE_CAP):
    """Unique video stems. Frames are '<stem>_f0001.jpg' -> strip the frame id.
    Media files only, so manifests like index.csv are not counted as videos."""
    seen, n = set(), 0
    for dirpath, _d, filenames in os.walk(root):
        for fn in filenames:
            if Path(fn).suffix.lower() not in MEDIA_EXT:
                continue
            n += 1
            st = Path(fn).stem
            if "_f" in st and st.rsplit("_f", 1)[-1].isdigit():
                st = st.rsplit("_f", 1)[0]
            seen.add(st)
            if n >= cap:
                return seen
    return seen

SUMMARY = {}
for name, s in STATS.items():
    stems = stems_of(s["path"])
    counts = collections.Counter(source_of(st) for st in stems)
    SUMMARY[name] = dict(stems=len(stems), counts=counts)
    print("=" * 78)
    print(f"{name}  -  {len(stems):,} unique video stems")
    for tag, c in counts.most_common():
        print(f"     {tag:<40}{c:>8,}  ({100*c/max(len(stems),1):5.1f}%)")
    ex = sorted(stems)[:6]
    print(f"     examples: {ex}")
    print()

## 5. Generator tags
AVGen-Bench files encode the generator as `avg_<model>_<hash>` — this recovers the exact model list for your paper.

In [ ]:
# ============================== 5. GENERATORS ================================
gens = collections.Counter()
for name, s in STATS.items():
    for st in stems_of(s["path"]):
        if st.startswith("avg_"):
            tail = st[4:]
            gens[tail.rsplit("_", 1)[0] if "_" in tail else tail] += 1

if gens:
    print(f"{len(gens)} distinct generators tagged\n")
    for g, c in gens.most_common():
        print(f"  {g:<44}{c:>7,}")
else:
    print("No 'avg_' tagged files found in the attached inputs.")

## 6. Provenance table for the paper
Paste this into your Data section, then fill the **Original source** column by opening each dataset page on Kaggle.

In [ ]:
# ============================== 6. FINAL TABLE ===============================
print(f"{'Kaggle dataset':<34}{'videos':>9}{'files':>11}   dominant source")
print("-" * 96)
for name, s in STATS.items():
    sm = SUMMARY[name]
    dom = sm["counts"].most_common(1)[0][0] if sm["counts"] else "-"
    print(f"{name:<34}{sm['stems']:>9,}{s['n']:>11,}   {dom}")

print("\n\nTO FILL IN MANUALLY (open each dataset page on Kaggle -> description/source):")
for name in STATS:
    print(f"  - {name:<34} original source: ______________________")

untagged = sum(sm["counts"].get("UNTAGGED (stock_pool / legacy_pool)", 0)
               for sm in SUMMARY.values())
total = sum(sm["stems"] for sm in SUMMARY.values()) or 1
print(f"\nUNTAGGED videos: {untagged:,} / {total:,} ({100*untagged/total:.1f}%)"
      "  <- these need provenance before publication")